In [1]:
# !pip install -U dspy-ai litellm openai
import os, json, re, textwrap, datetime
import dspy, litellm

In [2]:
import importlib.metadata

print("dspy-ai:", importlib.metadata.version("dspy"))
print("litellm:", importlib.metadata.version("litellm"))
print("openai:", importlib.metadata.version("openai"))


dspy-ai: 3.1.3
litellm: 1.81.15
openai: 2.15.0


In [3]:
philocal = dspy.LM(
    "openai/llama-local",              # nom arbitraire côté client
    api_base="http://localhost:8080/v1",
    api_key="dummy",                   # requis par certaines libs, peut être vide
    temperature=0.2
)

In [4]:
philocal("Dites bonjour, répondez en un mot")

['Bonjour!']

### Clé API et connexion LLM

#OpenAI
api_key = os.getenv("OPENAI_API_KEY")
lm = dspy.LM(
#model="gpt-5-mini-2025-08-07",          # modèle reconnu par l’API OpenAI
    model="gpt-5-mini",          # modèle reconnu par l’API OpenAI
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=1.0,
    cache=True,
    max_tokens=16000        # <- obligatoire ici pour désigner le backend
)
#dspy.settings.cache = None
dspy.settings.configure(lm=lm)
lm("Dites OK, répondez en un mot")

In [5]:
# DeepSeek
lm = dspy.LM(
    "openai/deepseek-chat",          # modèle DeepSeek côté API
    api_base="https://api.deepseek.com",  # ou https://api.deepseek.com/v1
    api_key=os.environ["DEEPSEEK_API_KEY"],
    model_type="chat",
)
dspy.settings.cache = None
dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=False)   
dspy.settings.configure(lm=lm)
lm("Dites bonjour, répondez en un mot")

['Bonjour']

# DSPy

## Paramètres

In [6]:
themes = [
    "informatique quantique", 
    "Advaita Vedanta", 
    "politique de l'enfant unique", 
    "relativité et éther",
    "Théorème des accroissements finis",
    "Voyage au bout de la nuit",
    "DSPy",
    "Le mécanisme d'attention dans les transformers",
    "Des souris et des hommes",
    "La commission Warren",
    "Dix petits nègres",
    "Comment j'ai mangé mon grand-père",
    "La conquête de l'Ouest et le capitalisme",
    "La théorie des nombres en cryptographie",
    "Histoire de l'écriture",
    "Temps et physique quantique",
    "La conquête du pouvoir de Benito",
    "La marche de boudienny",
    "Le fusil M56T, manuel d'entretien",
        "phénoménologie de la perception",
    "la mécanique céleste de Laplace",
    "la conscience chez les animaux",
    "les camps soviétiques du Goulag",
    "le réalisme magique en littérature",
    "le paradoxe d'Olbers",
    "la musique dodécaphonique",
    "la dialectique hégélienne",
    "les utopies technologiques",
    "la physique des trous noirs",
    "la mémoire collective et l'identité",
    "l’éthique de Spinoza",
    "les réseaux de neurones convolutifs",
    "les grands incendies de Rome",
    "la notion de sacrifice chez les Aztèques",
    "le mythe d'Orphée",
    "les accords de Bretton Woods",
    "la stratégie militaire de Napoléon",
    "la cognition incarnée",
    "la classification des particules élémentaires",
    "l'art brut",
    "la vie de Nikola Tesla",
    "les effets de la dopamine",
    "la poétique d’Aristote",
    "les codes QR et la logistique moderne",
    "le roman épistolaire",
    "la révolution cognitive du Paléolithique",
    "l’histoire de l’alchimie",
    "la logique floue",
    "les illusions d'optique",
    "les expériences de Milgram",
    "la cybernétique de Norbert Wiener",
    "la symbolique des couleurs",
    "le procès de Socrate",
    "la lecture rapide et ses limites",
    "la biologie synthétique",
    "le concept de karma",
    "la cartographie ancienne",
    "le langage des abeilles",
    "les protocoles de consensus en blockchain",
    "le nihilisme russe au XIXe siècle",
    "la thermodynamique statistique",
    "le théâtre de l'absurde",
    "les mathématiques de Gödel",
    "la métaphysique bouddhiste",
    "le chant diphonique",
    "la transmission orale des savoirs",
    "l'intelligence des corvidés",
    "le cinéma expressionniste allemand",
    "le syndrome de l'imposteur"
]

## Exemples

In [7]:
exemples = [{
    "theme": "libre",
    "mots_clefs": False,
    "titre": False,
    "date": False,
    "url": False,
    "texte": "Ne pas oublier de tailler la haie"
},
{
    "theme": "libre",
    "mots_clefs": True,
    "titre": False,
    "date": False,
    "url": False,
    "texte": "#jardin #urgent Ne pas oublier de tailler la haie"
},
{
    "theme": "libre",
    "mots_clefs": True,
    "titre": False,
    "date": True,
    "url": False,
    "texte": "#jardin #urgent Ne pas oublier de tailler la haie\n05/05/2005"
},
{
    "theme": "Poésie",
    "mots_clefs": False,
    "titre": False,
    "date": False,
    "url": False,
    "texte": "Rappelez-vous l'objet que nous vîmes, mon âme,\nCe beau matin d'été si doux:\nAu détour d'un sentier une charogne infâme\nSur un lit semé de cailloux,\nLe ventre en l'air, comme une femme lubrique,\nBrûlante et suant les poisons,\nOuvrait d'une façon nonchalante et cyniquen\nSon ventre plein d'exhalaisons.\nLe soleil rayonnait sur cette pourriture,\nComme afin de la cuire à point,\nEt de rendre au centuple à la grande Nature\nTout ce qu'ensemble elle avait joint;"
},
{
    "theme": "Poésie",
    "mots_clefs": True,
    "titre": False,
    "date": False,
    "url": False,
    "texte": "#Baudelaire #extrait #adolescence Rappelez-vous l'objet que nous vîmes, mon âme,\nCe beau matin d'été si doux:\nAu détour d'un sentier une charogne infâme\nSur un lit semé de cailloux,\nLe ventre en l'air, comme une femme lubrique,\nBrûlante et suant les poisons,\nOuvrait d'une façon nonchalante et cyniquen\nSon ventre plein d'exhalaisons.\nLe soleil rayonnait sur cette pourriture,\nComme afin de la cuire à point,\nEt de rendre au centuple à la grande Nature\nTout ce qu'ensemble elle avait joint;"
},
{
    "theme": "Poésie",
    "mots_clefs":True,
    "titre": True,
    "date": False,
    "url": False,
    "texte": "#Baudelaire #extrait #adolescence Titre: Une charogne\nRappelez-vous l'objet que nous vîmes, mon âme,\nCe beau matin d'été si doux:\nAu détour d'un sentier une charogne infâme\nSur un lit semé de cailloux,\nLe ventre en l'air, comme une femme lubrique,\nBrûlante et suant les poisons,\nOuvrait d'une façon nonchalante et cyniquen\nSon ventre plein d'exhalaisons.\nLe soleil rayonnait sur cette pourriture,\nComme afin de la cuire à point,\nEt de rendre au centuple à la grande Nature\nTout ce qu'ensemble elle avait joint;"
},
{
    "theme": "Poésie",
    "mots_clefs":True,
    "titre": True,
    "date": False,
    "url": True,
    "texte": "#Baudelaire #extrait #adolescence Titre: Une charogne\nRappelez-vous l'objet que nous vîmes, mon âme,\nCe beau matin d'été si doux:\nAu détour d'un sentier une charogne infâme\nSur un lit semé de cailloux,\nLe ventre en l'air, comme une femme lubrique,\nBrûlante et suant les poisons,\nOuvrait d'une façon nonchalante et cyniquen\nSon ventre plein d'exhalaisons.\nLe soleil rayonnait sur cette pourriture,\nComme afin de la cuire à point,\nEt de rendre au centuple à la grande Nature\nTout ce qu'ensemble elle avait joint;\n\nURL: mon_url.com"
},
{
    "theme": "Poésie",
    "mots_clefs":True,
    "titre": True,
    "date": True,
    "url": True,
    "texte": "#Baudelaire #extrait #adolescence Titre: Une charogne\nRappelez-vous l'objet que nous vîmes, mon âme,\nCe beau matin d'été si doux:\nAu détour d'un sentier une charogne infâme\nSur un lit semé de cailloux,\nLe ventre en l'air, comme une femme lubrique,\nBrûlante et suant les poisons,\nOuvrait d'une façon nonchalante et cyniquen\nSon ventre plein d'exhalaisons.\nLe soleil rayonnait sur cette pourriture,\nComme afin de la cuire à point,\nEt de rendre au centuple à la grande Nature\nTout ce qu'ensemble elle avait joint;\n\nURL: mon_url.com\nDate: 20/06/2020"
},
{
    "theme": "Histoire",
    "mots_clefs": False,
    "titre": False,
    "date": False,
    "url": False,
    "texte": "La guerre russo-japonaise (en japonais : 日露戦争 ; en russe : ру́сско-япóнская войнá) se déroule\ndu 8 février 1904 au 5 septembre 1905. Elle oppose l'Empire russe à l'empire du Japon,\nlequel, victorieux, obtient par le traité de Portsmouth la péninsule du Guandong et la moitié méridionale de l'île de Sakhaline.\n\nSur le plan militaire, ce conflit préfigure les guerres du XXe siècle par sa durée (un an et demi)[pourquoi ?], par les forces engagées et les pertes ainsi que par l'emploi des techniques les plus modernes de l'art de la guerre (logistique, lignes de communications et renseignements ; opérations combinées terrestres et maritimes ; durée de préparation des engagements)."
},
{
    "theme": "Histoire",
    "mots_clefs":True,
    "titre": False,
    "date": False,
    "url": False,
    "texte": "#guerre #Russie #Japon La guerre russo-japonaise (en japonais : 日露戦争 ; en russe : ру́сско-япóнская войнá) se déroule\ndu 8 février 1904 au 5 septembre 1905. Elle oppose l'Empire russe à l'empire du Japon,\nlequel, victorieux, obtient par le traité de Portsmouth la péninsule du Guandong et la moitié méridionale de l'île de Sakhaline.\n\nSur le plan militaire, ce conflit préfigure les guerres du XXe siècle par sa durée (un an et demi)[pourquoi ?], par les forces engagées et les pertes ainsi que par l'emploi des techniques les plus modernes de l'art de la guerre (logistique, lignes de communications et renseignements ; opérations combinées terrestres et maritimes ; durée de préparation des engagements)."
},
{
    "theme": "Histoire",
    "mots_clefs": False,
    "titre": True,
    "date": False,
    "url": False,
    "texte": "Titre: Guerre russo-japonaise\nLa guerre russo-japonaise (en japonais : 日露戦争 ; en russe : ру́сско-япóнская войнá) se déroule\ndu 8 février 1904 au 5 septembre 1905. Elle oppose l'Empire russe à l'empire du Japon,\nlequel, victorieux, obtient par le traité de Portsmouth la péninsule du Guandong et la moitié méridionale de l'île de Sakhaline.\n\nSur le plan militaire, ce conflit préfigure les guerres du XXe siècle par sa durée (un an et demi)[pourquoi ?], par les forces engagées et les pertes ainsi que par l'emploi des techniques les plus modernes de l'art de la guerre (logistique, lignes de communications et renseignements ; opérations combinées terrestres et maritimes ; durée de préparation des engagements)."
},
{
    "theme": "Histoire",
    "mots_clefs": False,
    "titre": True,
    "date": False,
    "url": False,
    "texte": "Titre: Guerre russo-japonaise\nLa guerre russo-japonaise (en japonais : 日露戦争 ; en russe : ру́сско-япóнская войнá) se déroule\ndu 8 février 1904 au 5 septembre 1905. Elle oppose l'Empire russe à l'empire du Japon,\nlequel, victorieux, obtient par le traité de Portsmouth la péninsule du Guandong et la moitié méridionale de l'île de Sakhaline.\n\nSur le plan militaire, ce conflit préfigure les guerres du XXe siècle par sa durée (un an et demi)[pourquoi ?], par les forces engagées et les pertes ainsi que par l'emploi des techniques les plus modernes de l'art de la guerre (logistique, lignes de communications et renseignements ; opérations combinées terrestres et maritimes ; durée de préparation des engagements)."
},
{
    "theme": "Histoire",
    "mots_clefs": True,
    "titre": True,
    "date": False,
    "url": False,
    "texte": "#guerre #Russie #Japon Titre: Guerre russo-japonaise\nLa guerre russo-japonaise (en japonais : 日露戦争 ; en russe : ру́сско-япóнская войнá) se déroule\ndu 8 février 1904 au 5 septembre 1905. Elle oppose l'Empire russe à l'empire du Japon,\nlequel, victorieux, obtient par le traité de Portsmouth la péninsule du Guandong et la moitié méridionale de l'île de Sakhaline.\n\nSur le plan militaire, ce conflit préfigure les guerres du XXe siècle par sa durée (un an et demi)[pourquoi ?], par les forces engagées et les pertes ainsi que par l'emploi des techniques les plus modernes de l'art de la guerre (logistique, lignes de communications et renseignements ; opérations combinées terrestres et maritimes ; durée de préparation des engagements)."
}
           ]


In [8]:
len(exemples)

13

In [9]:
champs_a_supprimer = ["expressions_clefs"]
nouveaux_exemples = [
    {k: v for k, v in item.items() if k not in champs_a_supprimer}
    for item in exemples
]
print(json.dumps(nouveaux_exemples, ensure_ascii=False, indent=2, sort_keys=True))

[
  {
    "date": false,
    "mots_clefs": false,
    "texte": "Ne pas oublier de tailler la haie",
    "theme": "libre",
    "titre": false,
    "url": false
  },
  {
    "date": false,
    "mots_clefs": true,
    "texte": "#jardin #urgent Ne pas oublier de tailler la haie",
    "theme": "libre",
    "titre": false,
    "url": false
  },
  {
    "date": true,
    "mots_clefs": true,
    "texte": "#jardin #urgent Ne pas oublier de tailler la haie\n05/05/2005",
    "theme": "libre",
    "titre": false,
    "url": false
  },
  {
    "date": false,
    "mots_clefs": false,
    "texte": "Rappelez-vous l'objet que nous vîmes, mon âme,\nCe beau matin d'été si doux:\nAu détour d'un sentier une charogne infâme\nSur un lit semé de cailloux,\nLe ventre en l'air, comme une femme lubrique,\nBrûlante et suant les poisons,\nOuvrait d'une façon nonchalante et cyniquen\nSon ventre plein d'exhalaisons.\nLe soleil rayonnait sur cette pourriture,\nComme afin de la cuire à point,\nEt de rendre au centuple

## Générateur DSPy de parangons

In [11]:
class GenerateNote(dspy.Signature):
    """Générez uniquement un texte en français conforme aux règles suivantes.

Entrées :
- theme
- mots_clefs (bool)
- titre (bool)
- date (bool)
- url (bool)

Format obligatoire, dans cet ordre exact :
1. si mots_clefs=True : une ligne de mots-clés exclusivement, sous la forme
#mot1 #mot2 #mot3
2. si titre=True : une ligne contenant uniquement le titre
3. si date=True : une ligne contenant uniquement une date au format JJ-MM-AAAA
4. si url=True : une ligne contenant uniquement une URL plausible
5. ensuite : le corps du texte

Règles :
- aucun libellé comme "Mots-clefs :", "Titre :", "Date :" ou "URL :"
- aucun markdown
- aucune puce
- aucune explication
- si un booléen vaut False, l'élément doit être absent
- les mots-clés doivent commencer par # et ne contenir aucun espace interne
- sortie = texte brut uniquement

Contraintes à respecter absolument:
- si mots_clefs=True, inclure quelque part dans la sortie une ligne contenant uniquement des mots-clés préfixés par #, par exemple :
#mot1 #mot2 #mot3
- ne jamais écrire 'Mots-clefs :' ni '**Mots-clefs**'
- si titre=True, inclure un titre sur une ligne distincte
- si date=True, inclure une date réaliste au format JJ-MM-AAAA
- si url=True, inclure une URL plausible
- si un booléen vaut False, l'élément correspondant doit être absent
- la position relative de ces éléments est libre
- sortie en texte brut uniquement
But final
Générer un texte permettant à un micro LLM comme Qwen3-0.6b de reconnaître la morphologie des notes qu'il devra traiter.
"""
    
    theme:str = dspy.InputField(desc="Thème principal de la note")
    mots_clefs: bool =  dspy.InputField(desc="Le texte est-il accompagné d'une liste de mots clefs")
    titre: bool = dspy.InputField(desc="Le texte est-il accompagné d'un titre")
    date: bool = dspy.InputField(desc="Le texte est-il accompagné d'une date")
    url: bool = dspy.InputField(desc="Le texte est-il accompagné d'une url")
    
    texte : str = dspy.OutputField(desc="Le texte lui-même qui être en accord avec le thème 'theme' ")
class NoteGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.Predict(GenerateNote)
    
    def forward(self, theme, mots_clefs, titre, date, url):
        result = self.generate(theme=theme, mots_clefs=mots_clefs, titre=titre, date=date,  url=url)
        print(f"{theme}")
        return {
            "texte": result.texte,
        }



## Comparateur de textes pour la métrique sémantique
Pour deux textes (text_a, text_b) produit UNE note globale 1..5
(thème + ton + style). 5 = très ressemblant, 1 = très différent.


In [12]:
# ---
# DSPy : comparaison 2 textes → score global 1..5, avec LM particulier
# Correction : ne jamais passer lm= à dspy.Predict ; utiliser un context manager.
# ---

import dspy

# 1) Définir un LM particulier (ok dspy.LM)
lm_comparateur = dspy.LM(
    model="gpt-4o-mini",   # ajustez le nom du modèle
    temperature=0.0,
    cache=True,
    max_tokens=128
)

# 2) Signature -> un seul entier 1..5
class CompareTextsGlobal(dspy.Signature):
    """
    Compare deux textes (text_a, text_b) et produit UNE note globale 1..5
    (thème + ton + style). 5 = très ressemblant, 1 = très différent.
    Sortie : un entier 1..5, sans commentaire.
    """
    text_a: str = dspy.InputField(desc="Premier texte")
    text_b: str = dspy.InputField(desc="Second texte")
    score: int  = dspy.OutputField(desc="Entier 1..5, score global")

# 3) Module : pas de lm passé à Predict ; on l'applique via un contexte au moment de l'appel
class TextComparatorGlobal(dspy.Module):
    def __init__(self, lm: dspy.LM | None = None):
        super().__init__()
        self.pred = dspy.Predict(CompareTextsGlobal)  # <- pas de lm ici
        self.lm = lm

    def forward(self, text_a: str, text_b: str) -> int:
        # On utilise le LM voulu via un context manager (pas sérialisé dans la requête)
        if self.lm is not None:
            with dspy.settings.context(lm=self.lm):
                out = self.pred(text_a=text_a, text_b=text_b)
        else:
            # Sinon, ça utilisera le LM global configuré ailleurs
            out = self.pred(text_a=text_a, text_b=text_b)

        # garde-fou : cast/clamp 1..5
        try:
            s = int(getattr(out, "score", 3))
        except Exception:
            s = 3
        return max(1, min(5, s))

# 4) Exemple d’utilisation avec LM local
comparator = TextComparatorGlobal(lm=lm_comparateur)

text1 = "Article expliquant la politique de l'enfant unique en Chine."
text2 = "Synthèse historique de la politique démographique chinoise au XXe siècle."

score = comparator.forward(text1, text2)
print("Score global (1..5):", score)


2026/03/11 10:50:35 WARNING dspy.primitives.module: Calling module.forward(...) on TextComparatorGlobal directly is discouraged. Please use module(...) instead.


Score global (1..5): 3


In [13]:
from typing import Any, Dict, Iterable, List, Optional, Tuple, Union
Json = Union[str, Dict[str, Any]]

compteur_global = 0;
def mentions_expr_as_input(text: str) -> bool:
    return bool(re.search(
        r"(fournir|donnez|veuillez|merci de).*(expression[_ -]?cl[ée]s?)",
        text, flags=re.I
    ))


def semantic_metric(
    gold: dict,
    pred: dict,
    trace: dict | None = None,
    pred_name: str | None = None,
    pred_trace: dict | None = None,
    *,
    w_semantic: float = 1.0,   # mettez 1.0 si vous n'utilisez pas l'expression
    w_expr: float = 0.0,
) -> float:
    """Score [0,1] basé sur le comparateur DSPy (1..5 → /5)."""
    gold_text = str((gold or {}).get("texte") or "")
    pred_text = str((pred or {}).get("texte") or "")

    if not gold_text or not pred_text:
        return 0.1  # neutre faible si vide

    if mentions_expr_as_input(pred_text):
        return 0.0  # rejet net
    try:
        # Utilisez l’opérateur d’appel, pas .forward (évite l’avertissement)
        score_15 = comparator(text_a=gold_text, text_b=pred_text)
        s_semantic = score_15 / 5.0
    except Exception:
        print(f"Erreur de comparaison entre {gold_text} et {pred_text}") 
        s_semantic = 0.6  # neutre moyen si échec LLM

    score = float(s_semantic)  # ici w_expr=0.0, donc score = s_semantic

    if isinstance(trace, dict):
        trace.update({
            "semantic_raw_1_5": score_15 if 'score_15' in locals() else None,
            "semantic": s_semantic,
            "final": score,
            "pred_name": pred_name,
        })
    if isinstance(pred_trace, dict):
        pred_trace.update({"final": score})

    global compteur_global
    compteur_global +=  1
    if compteur_global % 10 == 0:
        print(f"******************************************** au tour {compteur_global} score = {score}")
    return score


Notre but est d'abord d'obtenir un prompt amélioré par GEPA.
L'utilisation de gpt-4o-mini est cachée donc l'entrainement est très rapide.


In [14]:
from dspy.teleprompt import GEPA

trainset = [
    dspy.Example(
        theme=ex["theme"],
        texte=ex["texte"],
        mots_clefs=ex["mots_clefs"],
        titre=ex["titre"],
        date=ex["date"],
        url=ex["url"],
    ).with_inputs("theme","mots_clefs", "titre", "date", "url")
    for ex in exemples
]
print(len(trainset))

#for i, ex in enumerate(trainset):
#    try:
#        out = generator.forward(**ex.inputs())
#        print(f"✅ Exemple {i} OK")
#    except Exception as e:
#        print(f"❌ Erreur à l'exemple {i} : {e}")


# Initialisation et entraînement
generator = NoteGenerator()

teleprompter = dspy.GEPA(
    metric=semantic_metric,
    reflection_lm=lm,
    # Option A : profil auto
    #auto="heavy",

    # Option B : borne explicite (choisir l'une des deux lignes suivantes)
    #max_full_evals=1,          # très court (démonstration)
    max_metric_calls=10,    # borne sur le nb total d'appels métrique
    track_stats=True,
    track_best_outputs=True,
)
"""teleprompter = dspy.GEPA(
    metric=semantic_metric,
    reflection_lm=lm,
    auto="heavy",
    iterations=200,         # nombre d’itérations max
    patience=10,            # stop après 10 itérations sans progrès
    max_metric_calls=500,   # borne dure sur nb appels métrique
    track_stats=True,
    track_best_outputs=True
)"""

try:
    compiled_generator = teleprompter.compile(generator, trainset=trainset)
    compiled_generator.save("compiled_generator.json")
except RateLimitError as e:
    print("Une erreur est survenue lors de la compilation :", e)


2026/03/11 10:50:59 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 10 metric calls of the program. This amounts to 0.77 full evals on the train set.
2026/03/11 10:50:59 WARNING dspy.teleprompt.gepa.gepa: No valset provided; Using trainset as valset. This is useful as an inference-time scaling strategy where you want GEPA to find the best solutions for the provided tasks in the trainset, as it makes GEPA overfit prompts to the provided trainset. In order to ensure generalization and perform well on unseen tasks, please provide separate trainset and valset. Provide the smallest valset that is just large enough to match the downstream task distribution, while keeping trainset as large as possible.
2026/03/11 10:50:59 INFO dspy.teleprompt.gepa.gepa: Using 13 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just l

13


GEPA Optimization:   0%|          | 0/10 [00:00<?, ?rollouts/s]

Poésie
Poésie
Poésie
libre
libre
Poésie
libre
Poésie
Histoire
Histoire
Histoire
Histoire
Histoire
******************************************** au tour 10 score = 0.4


2026/03/11 10:51:20 INFO dspy.evaluate.evaluate: Average Metric: 3.8000000000000003 / 13 (29.2%)
2026/03/11 10:51:20 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.2923076923076923 over 13 / 13 examples
GEPA Optimization:   0%|          | 0/10 [00:20<?, ?rollouts/s]


In [15]:
compiled_generator = NoteGenerator()      # même classe que `generator`
compiled_generator.load("compiled_generator.json")

In [16]:
res = compiled_generator(theme="spiritualité",mots_clefs=True,titre=False,url=False,date=False)
print(res)
res = compiled_generator(theme="spiritualité",mots_clefs=False,titre=False,url=True,date=False)
print(res)
res = compiled_generator(theme="Bricolage",mots_clefs=False,titre=True,url=False,date=True)
print(res)


spiritualité
{'texte': "#spiritualité #méditation #conscience\n\nLa spiritualité est un chemin intérieur qui nous invite à explorer les profondeurs de notre être. Elle ne se limite pas à une religion ou une pratique spécifique, mais englobe une quête de sens, de connexion et de paix intérieure. Beaucoup trouvent dans la méditation un outil précieux pour calmer l'esprit et accéder à un état de présence. Cette recherche personnelle peut mener à une plus grande compassion envers soi-même et les autres, transformant notre perception du quotidien."}
spiritualité
{'texte': "https://www.voie-interieure.org/meditations-guidées\nLa spiritualité n'est pas une fuite du monde, mais une manière plus profonde de l'habiter. Elle invite à un recueillement silencieux, où le bruit des pensées s'apaise pour laisser place à une présence simple. C'est dans ce silence que l'on perçoit parfois un écho d'éternité, une paix qui ne dépend d'aucune circonstance extérieure. Cette quête est intime et personnelle ;

In [ ]:
prompt_gepa = """ Générez uniquement un texte en français conforme aux règles suivantes.

Entrées :
- theme
- mots_clefs (bool)
- titre (bool)
- date (bool)
- url (bool)

Format obligatoire, dans cet ordre exact :
1. si mots_clefs=True : une ligne de mots-clés exclusivement, sous la forme
#mot1 #mot2 #mot3
2. si titre=True : une ligne contenant uniquement le titre
3. si date=True : une ligne contenant uniquement une date au format JJ-MM-AAAA
4. si url=True : une ligne contenant uniquement une URL plausible
5. ensuite : le corps du texte

Règles :
- aucun libellé comme "Mots-clefs :", "Titre :", "Date :" ou "URL :"
- aucun markdown
- aucune puce
- aucune explication
- si un booléen vaut False, l'élément doit être absent
- les mots-clés doivent commencer par # et ne contenir aucun espace interne
- sortie = texte brut uniquement

Contraintes à respecter absolument:
- si mots_clefs=True, inclure quelque part dans la sortie une ligne contenant uniquement des mots-clés préfixés par #, par exemple :
#mot1 #mot2 #mot3
- ne jamais écrire 'Mots-clefs :' ni '**Mots-clefs**'
- si titre=True, inclure un titre sur une ligne distincte
- si date=True, inclure une date réaliste au format JJ-MM-AAAA
- si url=True, inclure une URL plausible
- si un booléen vaut False, l'élément correspondant doit être absent
- la position relative de ces éléments est libre
- sortie en texte brut uniquement
"""


In [ ]:
class SignatureEntreeCorpus(dspy.Signature):
    f"""Utiliser le prompt ci-dessous pour produire un texte.
    {prompt_gepa}
    Le modèle doit suivre ces instructions telles quelles.
    """

    theme:str = dspy.InputField(desc="Thème principal de la note")
    mots_clefs: bool =  dspy.InputField(desc="True si le texte est accompagné d'une liste de mots clefs")
    titre: bool = dspy.InputField(desc="True si le texte est accompagné d'un titre")
    date: bool = dspy.InputField(desc="True si le texte est  accompagné d'une date")
    url: bool = dspy.InputField(desc="True si le texte est  accompagné d'une url")
    
    texte : str = dspy.OutputField(desc="Le texte lui-même qui être en accord avec le thème 'theme' ")

    
class GenerationEntreeCorpus(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate = dspy.Predict(SignatureEntreeCorpus) 

    def forward(self, theme: str, mots_clefs: bool, titre: bool, date:bool, url:bool):
        result = self.generate(theme=theme, mots_clefs=mots_clefs, titre=titre, date=date, url=url)
        #print(f"{categorie}")
        #print(f"{result}")
        return {
            "texte": result.texte,
        }
        

    


In [ ]:
dspy.settings.cache = None
dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=False)   


In [ ]:
gen = GenerationEntreeCorpus()

In [ ]:
compiled_generator(theme="spiritualité",mots_clefs=True,titre=False,date=False,url=False)

In [ ]:
import time
import random

generated_textes = []
output_path = "corpus-100.txt"
nbtextes = 100
nbbatch = 5
stime = time.time()
with open(output_path, "w", encoding="utf-8") as f:
    for i in range(nbtextes):
        mc = random.choice([True, False])
        ti = random.choice([True, False])
        da = random.choice([True, False])
        ur = random.choice([True, False])
        th = random.choice(themes)
        if i % nbbatch == 0 and i != 0:
            
            print(f"{i} / {texte}  {(time.time()-stime)}")
            for texte in generated_textes:
                gtexte = f"{texte},\n"
                #print(f"{gtexte}")
                f.write(gtexte)
            generated_textes = []
            
#        texte = gen(theme=th,mots_clefs=mc,titre=ti,date=da,url=ur)
        texte = compiled_generator(theme=th,mots_clefs=mc,titre=ti,date=da,url=ur)
        generated_textes.append(texte)
        
print(f"Corpus sauvegardé en {(time.time()-stime)}s dans {output_path}")